# Phase 2: Generative AI & Prompt Engineering
This notebook is dedicated to exploring and refining prompts using groq API.

# 1-API Setup 


In [20]:
# INSTALL DEPENDENCIES
%pip install groq python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [21]:
# IMPORTS libraries 
import os
import pandas as pd
import sys
sys.path.append("Generative_AI") # Tell Python where to find api_config.py 
from api_config import client          
import warnings
warnings.filterwarnings("ignore")

In [22]:
# Test connection to Groq
try:
    test = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{"role": "user", "content": "Reply with OK only."}],
        max_completion_tokens=10 
    )
    print("Groq connected:", test.choices[0].message.content)

except Exception as e:
    print("Connection failed:", e)

Groq connected: OK


# 2- Prompt Function 


In [23]:
#   Sends a prompt to Groq (Llama) and returns the generated advice.
#    temperature=0 keeps responses consistent across all templates.

def generate_attrition_advice(prompt, temperature=0): 
    try:
        response = client.chat.completions.create(
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            max_completion_tokens=500
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"


# 3-LOAD DATASET


In [24]:
# Load the preprocessed dataset 
df = pd.read_csv("Dataset/preprocessed_data.csv")
print("Shape:", df.shape)
print(df.head())

Shape: (1470, 54)
        Age  Attrition  BusinessTravel  DailyRate  DistanceFromHome  \
0  0.446350          1               1   0.742527         -1.010909   
1  1.322365          0               2  -1.297775         -0.147150   
2  0.008343          1               1   1.414363         -0.887515   
3 -0.429664          0               2   1.461466         -0.764121   
4 -1.086676          0               1  -0.524295         -0.887515   

   Education  EnvironmentSatisfaction  Gender  HourlyRate  JobInvolvement  \
0          2                        2  Female    1.383138               3   
1          1                        3    Male   -0.240677               2   
2          2                        4    Male    1.284725               2   
3          4                        4  Female   -0.486709               3   
4          1                        1    Male   -1.274014               3   

   ...  JobRole_Sales Representative  MaritalStatus_Divorced  \
0  ...                      

In [25]:
# SAVE EXAMPLE OUTPUTS FCUNCTION
def save_example_output(template_name, case_id, response):
    """Saves a response to Generative_AI/example_outputs/"""
    os.makedirs("Generative_AI/example_outputs", exist_ok=True)
    filename = f"Generative_AI/example_outputs/{template_name}_case{case_id}.txt"
    with open(filename, "w") as f:
        f.write(response)
    print(f"Saved: {filename}")


# 4- PROMPT templates

## Template 1 – Basic Explanation (T1)

### Intended Use Case
This template is designed to generate a simple explanation of the attrition prediction based on employee features. It is used as a baseline to evaluate how the model performs with minimal guidance.

### Design Rationale
The purpose of this template is to observe the model’s natural reasoning ability without enforcing structure or constraints. It helps establish a comparison point against more structured and advanced prompts.

### Prompt Structure
You are an HR assistant.

Employee profile:
{features}

Prediction: Attrition = {prediction}

Explain why the employee may leave the company.

### Example Input
Employee profile:
Age: 41  
BusinessTravel: Travel_Rarely  
Department: Sales  
DistanceFromHome: 1  
EducationField: Life Sciences  
JobRole: Sales Executive  
MonthlyIncome: 5993  
OverTime: Yes  
JobSatisfaction: 4  
WorkLifeBalance: 1  
YearsAtCompany: 6  

Prediction: Attrition = Yes

### Example Output
Based on the employee profile, I would explain the prediction of Attrition = Yes as follows:

Although the employee has a relatively high JobSatisfaction rating of 4, which might suggest they are content with their job, there are other factors that could be contributing to their predicted attrition.

One potential reason is the employee's WorkLifeBalance rating of 1, which is very low. This suggests that the employee may be experiencing significant difficulties in balancing their work and personal life, which could lead to burnout and dissatisfaction.

Additionally, the employee does work overtime (OverTime: Yes), which could be a sign of an unsustainable workload or poor work-life balance.

It's also worth noting that the employee travels rarely (BusinessTravel: Travel_Rarely) and lives close to the workplace (DistanceFromHome: 1), which might not be a significant contributor to attrition.

However, with 6 years of service (YearsAtCompany: 6) and a relatively high MonthlyIncome (5993), it's possible that the employee may be looking for new challenges or opportunities that are not available in their current role.

Considering these factors, the prediction of Attrition = Yes may be driven by the employee's poor WorkLifeBalance and potential burnout, which could be outweighing their job satisfaction. Further discussion or analysis would be needed to confirm the reasons behind the prediction.



### Assumptions & Limitations
- Assumes the model can infer relationships between features and attrition without guidance  
- May produce vague or generic responses due to lack of structure  
- Does not enforce actionable recommendations  

## Template 2 – Structured HR Analysis (T2)

### Intended Use Case
This template is designed to provide a structured and detailed HR analysis of the attrition prediction, including key factors, risks, and recommended actions.

### Design Rationale
This template introduces a structured format to guide the model toward generating more organized and actionable insights. By explicitly requesting key factors, risks, and recommendations, the output becomes more useful for decision-making compared to a basic explanation.

### Prompt Structure
You are an HR expert analyzing employee attrition risk.

Employee profile:
{features}

Prediction: Attrition = {prediction}

Provide:
1. Key factors contributing to this prediction
2. Risks associated with this employee leaving
3. Recommended actions to retain this employee

Use clear bullet points.

### Example Input
Employee profile:
Age: 37  
BusinessTravel: Travel_Rarely  
Department: Research & Development  
DistanceFromHome: 2  
EducationField: Other  
JobRole: Laboratory Technician  
MonthlyIncome: 2090  
OverTime: Yes  
JobSatisfaction: 2  
WorkLifeBalance: 3  
YearsAtCompany: 0  

Prediction: Attrition = Yes

### Example Output
Based on the employee profile, here is the analysis:

**1. Main factors behind this prediction:**

* **Low JobSatisfaction (2)**: The employee is not satisfied with their job, which is a strong indicator of attrition risk.
* **Low MonthlyIncome (2090)**: The employee's income is relatively low, which may lead to dissatisfaction and motivation to look for better-paying opportunities.
* **OverTime (Yes)**: The employee is required to work overtime, which can lead to burnout and negatively impact work-life balance.
* **Young tenure (YearsAtCompany: 0)**: The employee is new to the company, which makes them more vulnerable to attrition risk.

**2. Possible risk level:**
* **High Risk**: The combination of low job satisfaction, relatively low income, and overtime requirements suggests a high risk of attrition.

**3. Recommended HR actions:**
* **Retention interview**: Conduct a thorough interview with the employee to understand the root causes of their dissatisfaction and address their concerns.
* **Salary review**: Consider a salary adjustment to bring the employee's income more in line with industry standards.
* **Work-life balance improvement**: Explore options to reduce or manage overtime requirements to improve the employee's work-life balance.
* **Job enrichment**: Provide opportunities for growth and development within the Laboratory Technician role or consider lateral moves within the Research & Development department.
* **Onboarding and support**: Ensure the employee receives comprehensive onboarding and ongoing support to help them settle into their new role and feel more engaged with the company.


### Assumptions & Limitations
- Assumes structured prompts improve clarity and usefulness  
- May still include generic recommendations  
- Does not enforce strict personalization

## Template 3 – Manager-Friendly Explanation (T3)

### Intended Use Case
This template is designed to generate a simple and clear explanation for non-technical managers who need to understand attrition predictions without technical details.

### Design Rationale
This template focuses on readability and clarity by simplifying the explanation and avoiding technical language. It helps evaluate how well the model communicates insights to non-expert stakeholders.

### Prompt Structure
You are explaining the attrition prediction to a non-technical manager.

Employee profile:
{features}

Prediction: Attrition = {prediction}

Explain in simple language:
- Why this employee may stay or leave
- What the company should do next

Avoid technical terms.

### Example Input
Employee profile:
Age: 49  
BusinessTravel: Travel_Frequently  
Department: Research & Development  
DistanceFromHome: 8  
EducationField: Life Sciences  
JobRole: Research Scientist  
MonthlyIncome: 5130  
OverTime: No  
JobSatisfaction: 3  
WorkLifeBalance: 3  
YearsAtCompany: 10  

Prediction: Attrition = No

### Example Output
Let me break down the prediction in simple terms:

**Why this employee may stay:**
Our analysis suggests that this employee is likely to stay with the company. Here are a few reasons why:

* They have been with the company for 10 years, which indicates a level of loyalty and commitment.
* They have a relatively high monthly income of $5130, which suggests that they are fairly compensated for their work.
* Although their job satisfaction and work-life balance are only average (rated 3 out of 5), these factors aren't strong indicators of attrition in this case.

**Why this employee may leave (but we're not predicting that):**
While we're not predicting attrition, there are a few factors that could potentially contribute to turnover:

* The employee travels frequently for business, which can be stressful and impact work-life balance.
* They are 49 years old, which is an age where some employees might start thinking about retirement or exploring new opportunities.

**What the company should do next:**
Based on our analysis, here are some recommendations:

* Continue to monitor the employee's job satisfaction and work-life balance to ensure that these factors don't become a concern.
* Consider offering opportunities for growth and development within the Research & Development department to keep the employee engaged and motivated.
* Since the employee travels frequently, consider offering support or resources to help them manage their work-life balance while on the road.

Overall, our prediction suggests that this employee is likely to stay with the company, but it's always important to stay proactive and address any potential concerns before they become major issues.

### Assumptions & Limitations
- Assumes simpler language improves understanding  
- May oversimplify important details  
- Limited depth compared to structured analysis

## Template 4 – Personalized Retention Advice (T4)

### Intended Use Case
This template is designed to generate personalized and actionable retention advice based on individual employee characteristics.

### Design Rationale
This template aims to produce highly relevant and personalized outputs by focusing on individual features and requiring practical recommendations. It is expected to outperform other templates in relevance and usefulness.

### Prompt Structure
You are an HR consultant providing personalized advice.

Employee profile:
{features}

Prediction: Attrition = {prediction}

Based only on the provided employee data:
- Identify the most important individual risk factors
- Explain how they relate to attrition
- Provide practical and personalized recommendations
- Avoid unsupported assumptions

Be specific and realistic.

### Example Input
Employee profile:
Age: 27  
BusinessTravel: Travel_Rarely  
Department: Research & Development  
DistanceFromHome: 2  
EducationField: Medical  
JobRole: Research Scientist  
MonthlyIncome: 3468  
OverTime: Yes  
JobSatisfaction: 2  
WorkLifeBalance: 3  
YearsAtCompany: 2  

Prediction: Attrition = Yes


Prediction: Attrition = Yes

### Example Output
This employee is at risk of leaving due to a combination of low job satisfaction and overtime work. Although their work-life balance appears moderate, the additional workload from overtime may reduce overall satisfaction.

Since the employee is relatively early in their career with the company, they may also be exploring better opportunities.

To improve retention, the company should focus on reducing overtime, improving job satisfaction through feedback and recognition, and providing clear career development opportunities tailored to this employee.

### Assumptions & Limitations
- Assumes personalization improves relevance and decision-making  
- Relies only on provided features (no external context)  
- May still generate assumptions if input data is limited

# 6- Testing 

In [26]:
# get llama response
MODEL_NAME = "meta-llama/llama-4-scout-17b-16e-instruct"

def get_ai_response(prompt, max_tokens=350):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=max_tokens,
        temperature=0.3
    )
    return response.choices[0].message.content


templates = {
    "T1_Basic_Explanation": """
You are an HR assistant.

Employee profile:
{features}

Prediction: Attrition = {prediction}

Explain why the employee may be predicted as {prediction}.
Keep the explanation clear and relevant to the given employee data.
""",

    "T2_Structured_HR_Analysis": """
You are an HR expert analyzing employee attrition risk.

Employee profile:
{features}

Prediction: Attrition = {prediction}

Provide:
1. Main factors behind this prediction
2. Possible risk level
3. Recommended HR actions

Use clear bullet points.
""",

    "T3_Manager_Friendly_Explanation": """
You are explaining the attrition prediction to a non-technical manager.

Employee profile:
{features}

Prediction: Attrition = {prediction}

Explain in simple language:
- Why this employee may stay or leave
- What the company should do next

Avoid machine learning jargon.
""",

    "T4_Personalized_Retention_Advice": """
You are an HR consultant giving personalized retention advice.

Employee profile:
{features}

Prediction: Attrition = {prediction}

Based only on the provided employee data:
- Identify the most important individual risk factors
- Explain how they relate to attrition
- Give practical and personalized recommendations
- Avoid making unsupported assumptions

Be specific, realistic, and professional.
"""
}

In [28]:
#choose 3 rows from the dataset randomly

selected_cols = [
    "Age",
    "BusinessTravel",
    "Department",
    "DistanceFromHome",
    "EducationField",
    "JobRole",
    "MonthlyIncome",
    "OverTime",
    "JobSatisfaction",
    "WorkLifeBalance",
    "YearsAtCompany"
]
df = pd.read_csv("Dataset/Raw_Kaggle_HR_Employee_Attrition_data.csv")
sample_df = df.sample(n=3, random_state=42)
def row_to_features(row):
    return "\n".join([f"{col}: {row[col]}" for col in selected_cols])
test_cases = []

for i, row in sample_df.iterrows():
    test_cases.append({
        "case_id": f"Case {len(test_cases)+1}",
        "prediction": row["Attrition"],  # assumes column name is Attrition
        "features": row_to_features(row)
    })

In [30]:
#get & save results
results = []

for template_name, template_text in templates.items():
    for case in test_cases:
        prompt = template_text.format(
            features=case["features"],
            prediction=case["prediction"]
        )

        output = get_ai_response(prompt)

        results.append({
            "Template": template_name,
            "Case": case["case_id"],
            "Prediction": case["prediction"],
            "Output": output,
            "Word Count": len(output.split())
        })

results_df = pd.DataFrame(results)
results_df

results_df.to_csv("Generative_AI_Output_Comparison.csv", index=False)

In [ ]:
# TEMP CELL: Generate example outputs for documentation

templates = {
    "T1_Basic_Explanation": """
You are an HR assistant.

Employee profile:
{features}

Prediction: Attrition = {prediction}

Explain why the employee may be predicted as {prediction}.
Keep the explanation clear and relevant to the given employee data.
""",

    "T2_Structured_HR_Analysis": """
You are an HR expert analyzing employee attrition risk.

Employee profile:
{features}

Prediction: Attrition = {prediction}

Provide:
1. Main factors behind this prediction
2. Possible risk level
3. Recommended HR actions

Use clear bullet points.
""",

    "T3_Manager_Friendly_Explanation": """
You are explaining the attrition prediction to a non-technical manager.

Employee profile:
{features}

Prediction: Attrition = {prediction}

Explain in simple language:
- Why this employee may stay or leave
- What the company should do next

Avoid machine learning jargon.
""",

    "T4_Personalized_Retention_Advice": """
You are an HR consultant giving personalized retention advice.

Employee profile:
{features}

Prediction: Attrition = {prediction}

Based only on the provided employee data:
- Identify the most important individual risk factors
- Explain how they relate to attrition
- Give practical and personalized recommendations
- Avoid making unsupported assumptions

Be specific, realistic, and professional.
"""
}

# TEMP CELL: Use fixed Example Input

# TEMP CELL: Generate example outputs for documentation

features = """Employee profile:
Age: 27  
BusinessTravel: Travel_Rarely  
Department: Research & Development  
DistanceFromHome: 2  
EducationField: Medical  
JobRole: Research Scientist  
MonthlyIncome: 3468  
OverTime: Yes  
JobSatisfaction: 2  
WorkLifeBalance: 3  
YearsAtCompany: 2"""

prediction = "Yes"

template = templates["T3_Manager_Friendly_Explanation"]

prompt = template.format(
    features=features,
    prediction=prediction
)

output = get_ai_response(prompt)

print("### Example Input")
print(f"""Employee profile:
{features}

Prediction: Attrition = {prediction}""")

print("\n### Example Output")
print(output)



### Example Input
Employee profile:
Age: 49  
BusinessTravel: Travel_Frequently  
Department: Research & Development  
DistanceFromHome: 8  
EducationField: Life Sciences  
JobRole: Research Scientist  
MonthlyIncome: 5130  
OverTime: No  
JobSatisfaction: 3  
WorkLifeBalance: 3  
YearsAtCompany: 10 

Prediction: Attrition = NO

### Example Output
Let me break down the prediction in simple terms:

**Why this employee may stay:**
Our analysis suggests that this employee is likely to stay with the company. Here are a few reasons why:

* They have been with the company for 10 years, which indicates a level of loyalty and commitment.
* They have a relatively high monthly income of $5130, which suggests that they are fairly compensated for their work.
* Although their job satisfaction and work-life balance are only average (rated 3 out of 5), these factors aren't strong indicators of attrition in this case.

**Why this employee may leave (but we're not predicting that):**
While we're not pr